# mangaviewer × Colab 파이프라인

로컬 에뮬레이터 테스트 루프와 코랩(A100)을 연결한다. 로컬에서 프로브를 돌리고 → 인박스로 올리고 → 코랩이 빌드/테스트/A100 분석을 하고 → 결과를 회수한다.

**사용법**
1. 런타임 → 런타임 유형 변경 → **A100 GPU** 선택
2. 위에서부터 전부 실행 (Ctrl+F9)
3. 산출물: `results.tar.gz`(분석/로그), `artifacts.tar.gz`(APK)
4. 로컬 회수: `colab/pull_results.ps1`

> 빌드/테스트만 할 땐 **CPU 런타임**(CCU 안 씀). IMPROVE 구간만 GPU가 필요하다.

In [ ]:
#@title 1) 설정 { display-mode: "form" }
REPO_URL = "https://github.com/ad2das/mangaviewer.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
INBOX_BRANCH = "colab-inbox"  #@param {type:"string"}
OUTBOX_BRANCH = "colab-outbox"  #@param {type:"string"}
RUN_BUILD = True  #@param {type:"boolean"}
RUN_UNIT_TESTS = True  #@param {type:"boolean"}
RUN_TOOLS_TESTS = True  #@param {type:"boolean"}
RUN_IMPROVE = True  #@param {type:"boolean"}
CHAT_UI = False  #@param {type:"boolean"}
MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ"  #@param {type:"string"}
MAX_MODEL_LEN = 32768  #@param {type:"integer"}

import pathlib
import subprocess

WORK = "/content/mw"
REPO_DIR = f"{WORK}/repo"
INBOX_DIR = f"{WORK}/inbox"
OUT_DIR = f"{WORK}/out"
SDK_DIR = f"{WORK}/android-sdk"
for d in (REPO_DIR, INBOX_DIR, OUT_DIR, SDK_DIR, f"{OUT_DIR}/logs", f"{OUT_DIR}/artifacts"):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
gpu_name = gpu.stdout.strip() if gpu.returncode == 0 else ""
print("GPU:", gpu_name or "(none)")

if RUN_IMPROVE and not gpu_name:
    print("GPU 없음 -> RUN_IMPROVE 자동 비활성화. A100 런타임으로 바꾸고 처음부터 다시 실행해라.")
    RUN_IMPROVE = False

env_lines = [
    f'MW_REPO="{REPO_DIR}"',
    f'MW_INBOX="{INBOX_DIR}"',
    f'MW_OUT="{OUT_DIR}"',
    f'MW_SDK="{SDK_DIR}"',
    f'MW_MODEL="{MODEL_ID}"',
    f'MW_MAX_LEN="{MAX_MODEL_LEN}"',
    f'MW_REPO_URL="{REPO_URL}"',
    f'MW_REPO_BRANCH="{REPO_BRANCH}"',
    f'MW_INBOX_BRANCH="{INBOX_BRANCH}"',
    f'MW_OUTBOX_BRANCH="{OUTBOX_BRANCH}"',
    f'MW_RUN_BUILD={int(RUN_BUILD)}',
    f'MW_RUN_UNIT_TESTS={int(RUN_UNIT_TESTS)}',
    f'MW_RUN_TOOLS_TESTS={int(RUN_TOOLS_TESTS)}',
    f'MW_RUN_IMPROVE={int(RUN_IMPROVE)}',
    f'MW_CHAT_UI={int(CHAT_UI)}',
]
pathlib.Path("/content/mw-config.env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print("config written")

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env

# JDK 17
if ! java -version 2>&1 | grep -q 'version "17'; then
  apt-get -qq update >/dev/null 2>&1
  apt-get -qq install -y openjdk-17-jdk-headless unzip >/dev/null 2>&1
fi
JAVA_HOME="$(dirname "$(dirname "$(readlink -f "$(command -v java)")")")"
grep -q '^JAVA_HOME=' /content/mw-config.env || echo "JAVA_HOME=$JAVA_HOME" >> /content/mw-config.env
java -version 2>&1 | head -n 1

# Android cmdline-tools
if [ ! -x "$MW_SDK/cmdline-tools/latest/bin/sdkmanager" ]; then
  mkdir -p "$MW_SDK/cmdline-tools"
  curl -sSLo /tmp/cmdtools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
  unzip -q -o /tmp/cmdtools.zip -d "$MW_SDK/cmdline-tools"
  mv "$MW_SDK/cmdline-tools/cmdline-tools" "$MW_SDK/cmdline-tools/latest" 2>/dev/null || true
fi
export ANDROID_HOME="$MW_SDK"
export PATH="$MW_SDK/cmdline-tools/latest/bin:$MW_SDK/platform-tools:$PATH"

yes | sdkmanager --licenses >/dev/null 2>&1 || true
sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" "ndk;27.2.12479018" "cmake;3.22.1" >/tmp/sdkmanager.log 2>&1
echo "SDK components:"; ls "$MW_SDK/ndk" "$MW_SDK/cmake" 2>/dev/null

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env

if [ -d "$MW_REPO/.git" ]; then
  cd "$MW_REPO"
  git fetch --depth 1 origin "$MW_REPO_BRANCH" && git checkout -q -f FETCH_HEAD
else
  git clone --depth 1 --branch "$MW_REPO_BRANCH" "$MW_REPO_URL" "$MW_REPO" && cd "$MW_REPO"
fi
echo "HEAD: $(git log --oneline -1)"

# inbox (test artifacts pushed from the local machine)
rm -rf "$MW_INBOX"; mkdir -p "$MW_INBOX"
if git fetch --depth 1 origin "$MW_INBOX_BRANCH" 2>/dev/null; then
  if git archive --format=zip FETCH_HEAD inbox.zip > /content/inbox.zip 2>/dev/null && [ -s /content/inbox.zip ]; then
    (cd "$MW_INBOX" && unzip -q /content/inbox.zip)
    echo "inbox unpacked:"; find "$MW_INBOX" -maxdepth 3 -type f | head -n 20
  else
    echo "inbox branch present but no inbox.zip"
  fi
else
  echo "no $MW_INBOX_BRANCH branch - inbox empty"
fi

# optional firebase config: drag google_services.xml into the Colab file browser,
# or place build/google_services.xml inside the inbox zip.
GS_SRC=""
[ -f /content/google_services.xml ] && GS_SRC=/content/google_services.xml
if [ -z "$GS_SRC" ] && [ -f "$MW_INBOX/build/google_services.xml" ]; then
  GS_SRC="$MW_INBOX/build/google_services.xml"
fi
if [ -n "$GS_SRC" ]; then
  cp "$GS_SRC" "$MW_REPO/app/src/main/res/values/google_services.xml"
  echo "google_services.xml restored from $GS_SRC"
else
  echo "google_services.xml absent - viewer/build fine, Firebase login disabled"
fi

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env
[ "$MW_RUN_BUILD" = "1" ] || { echo "RUN_BUILD=false - skip"; exit 0; }

cd "$MW_REPO"
echo "sdk.dir=$MW_SDK" > local.properties
chmod +x gradlew
./gradlew --console=plain :app:assembleDebug :app:assembleDebugAndroidTest > "$MW_OUT/logs/gradle-build.log" 2>&1
rc=$?
tail -n 25 "$MW_OUT/logs/gradle-build.log"
echo "gradle rc=$rc"
if [ $rc -eq 0 ]; then echo "build=OK" >> /content/mw-status.env; else echo "build=FAILED" >> /content/mw-status.env; fi
find app/build/outputs/apk -name "*.apk" -exec cp {} "$MW_OUT/artifacts/" \; 2>/dev/null
ls -la "$MW_OUT/artifacts/"

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env
cd "$MW_REPO"

if [ "$MW_RUN_UNIT_TESTS" = "1" ]; then
  ./gradlew --console=plain test > "$MW_OUT/logs/gradle-tests.log" 2>&1
  rc=$?
  tail -n 15 "$MW_OUT/logs/gradle-tests.log"
  if [ $rc -eq 0 ]; then echo "unit_tests=OK" >> /content/mw-status.env; else echo "unit_tests=FAILED" >> /content/mw-status.env; fi
fi

if [ "$MW_RUN_TOOLS_TESTS" = "1" ]; then
  pip -q install --no-input pytest numpy pillow >/dev/null 2>&1
  python -m pytest tools/ -q > "$MW_OUT/logs/tools-pytest.log" 2>&1
  rc=$?
  tail -n 15 "$MW_OUT/logs/tools-pytest.log"
  if [ $rc -eq 0 ]; then echo "tools_tests=OK" >> /content/mw-status.env; else echo "tools_tests=FAILED" >> /content/mw-status.env; fi
fi

---
## A100 구간

여기서부터 GPU(CCU)를 소모한다. 빌드/테스트만 필요하면 아래 셀들은 건너뛰어도 된다.

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env
[ "$MW_RUN_IMPROVE" = "1" ] || { echo "RUN_IMPROVE=false - skip"; exit 0; }
pip -q install --no-input vllm openai > /tmp/vllm-install.log 2>&1
rc=$?
echo "pip rc=$rc"
if [ $rc -ne 0 ]; then tail -n 30 /tmp/vllm-install.log; fi
python -c "import vllm; print('vllm', vllm.__version__)" 2>&1 | tail -n 2

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env
[ "$MW_RUN_IMPROVE" = "1" ] || { echo "RUN_IMPROVE=false - skip"; exit 0; }

if curl -s http://127.0.0.1:8000/v1/models >/dev/null 2>&1; then echo "vLLM already running"; exit 0; fi

nohup python -m vllm.entrypoints.openai.api_server \
  --model "$MW_MODEL" --served-model-name coder \
  --max-model-len "$MW_MAX_LEN" --gpu-memory-utilization 0.92 \
  --port 8000 --disable-log-requests > /content/vllm.log 2>&1 &
echo $! > /content/vllm.pid
echo "loading $MW_MODEL (first run includes download, a few minutes)..."
for i in $(seq 1 180); do
  if curl -s http://127.0.0.1:8000/v1/models >/dev/null 2>&1; then echo "vLLM ready (${i}x5s)"; break; fi
  sleep 5
done
curl -s http://127.0.0.1:8000/v1/models | head -c 300; echo
tail -n 5 /content/vllm.log

In [ ]:
#@title 7c) 인박스 태스크 분석 -> 분석문/패치 생성
import json, pathlib, re, subprocess, time
from openai import OpenAI

SYSTEM_PROMPT = (
    "You are a senior Android engineer embedded in the 'mangaviewer' project "
    "(Kotlin + Jetpack Compose + native C++/GLES viewer via NDK; modules: app, core, viewer, "
    "viewer-content, engine-api, engine-v2, data, source-ntk, source-wfwf, source-newxtoon, source-goodtoon). "
    "The team validates viewer behavior with on-device instrumentation probes (frame cadence, resident "
    "tiles, work ownership, memory) and host-side python contracts. "
    "You receive an objective, questions, repo context, and probe attachments. "
    "Answer in Korean, in exactly this structure:\n"
    "## 진단 - concise diagnosis grounded in the given evidence (cite specific numbers/lines)\n"
    "## 수정 제안 - the single most defensible fix, with ONE unified diff in a ```diff block "
    "(paths relative to repo root, minimal change)\n"
    "## 검증 계획 - how to verify using the project's existing tools (gradle tests, emulator probes, host contracts)\n"
    "No generic advice. If evidence is insufficient, say exactly what is missing."
)

if not RUN_IMPROVE:
    print("RUN_IMPROVE=false - skip")
else:
    client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="colab")
    tasks_root = pathlib.Path(INBOX_DIR) / "tasks"
    analysis_root = pathlib.Path(OUT_DIR) / "analysis"
    analysis_root.mkdir(parents=True, exist_ok=True)

    def clip(text, limit):
        if not text:
            return ""
        return text if len(text) <= limit else text[:limit] + "\n...[truncated]..."

    def repo_context(task, task_dir):
        repo = pathlib.Path(REPO_DIR)
        tree = subprocess.run(["git", "-C", str(repo), "ls-files"],
                              capture_output=True, text=True).stdout
        shallow = [l for l in tree.splitlines() if l.count("/") <= 2]
        parts = ["# repo tree (top levels)\n" + "\n".join(shallow[:200])]
        budget = 40000
        for rel in (task.get("focus_files") or []):
            local = task_dir / "files" / rel
            p = local if local.is_file() else repo / rel
            if not p.is_file():
                continue
            body = clip(p.read_text(encoding="utf-8", errors="replace"), 8000)
            parts.append(f"\n## {rel}\n```\n{body}\n```")
            budget -= len(body)
            if budget <= 0:
                break
        return "\n".join(parts)

    def attachments(task_dir):
        parts, budget = [], 30000
        for f in sorted(task_dir.iterdir()):
            if f.name in ("task.json", "worktree.diff") or not f.is_file():
                continue
            body = clip(f.read_text(encoding="utf-8", errors="replace"), 12000)
            parts.append(f"\n## attachment: {f.name}\n```\n{body}\n```")
            budget -= len(body)
            if budget <= 0:
                break
        return "\n".join(parts)

    if not tasks_root.exists() or not any(p.is_dir() for p in tasks_root.iterdir()):
        print("인박스에 태스크 없음 - 로컬에서 colab/push_inputs.ps1 먼저 실행")
    else:
        for task_dir in sorted(p for p in tasks_root.iterdir() if p.is_dir()):
            task = json.loads((task_dir / "task.json").read_text(encoding="utf-8-sig"))
            questions = "\n".join(f"- {q}" for q in task.get("questions", []))
            diff_path = task_dir / "worktree.diff"
            diff_text = clip(diff_path.read_text(encoding="utf-8", errors="replace"), 20000) if diff_path.exists() else ""
            user = (
                f"## objective\n{task.get('objective', '')}\n\n"
                f"## questions\n{questions or '- (none)'}\n\n"
                f"## worktree diff\n```diff\n{diff_text}\n```\n\n"
                + repo_context(task, task_dir) + "\n" + attachments(task_dir)
            )
            messages = [{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user}]
            text = None
            for attempt in (1, 2):
                try:
                    r = client.chat.completions.create(model="coder", messages=messages,
                                                       temperature=0.2, max_tokens=8000)
                    text = r.choices[0].message.content
                    break
                except Exception as e:
                    print(f"{task_dir.name}: attempt {attempt} failed: {e}")
                    time.sleep(5)
            if not text:
                print(f"{task_dir.name}: FAILED")
                continue
            out = analysis_root / task_dir.name
            out.mkdir(parents=True, exist_ok=True)
            (out / "analysis.md").write_text(text, encoding="utf-8")
            m = re.search(r"```diff\n(.*?)```", text, re.S)
            if m:
                (out / "proposal.patch").write_text(m.group(1), encoding="utf-8")
            print(f"{task_dir.name}: {len(text)} chars, patch={'yes' if m else 'no'}")

In [ ]:
#@title 7d) (선택) A100 코더 채팅 UI
if RUN_IMPROVE and CHAT_UI:
    import subprocess
    subprocess.run("pip -q install --no-input gradio", shell=True)
    import gradio as gr
    from openai import OpenAI
    from google.colab import output as colab_output

    cli = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="colab")

    def chat(message, history):
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
        for h in history:
            msgs.append({"role": "user", "content": h[0]})
            msgs.append({"role": "assistant", "content": h[1]})
        msgs.append({"role": "user", "content": message})
        r = cli.chat.completions.create(model="coder", messages=msgs,
                                        temperature=0.3, max_tokens=4000)
        return r.choices[0].message.content

    demo = gr.ChatInterface(chat, title="mangaviewer coder (A100)")
    demo.launch(share=False, inline=False)
    colab_output.serve_kernel_port_as_iframe(7860, height=700)
else:
    print("CHAT_UI=false 또는 RUN_IMPROVE=false - skip")

In [ ]:
%%bash
set -uo pipefail
source /content/mw-config.env
cd "$MW_OUT"

python - <<'PY'
import datetime, json, os, pathlib, subprocess
out = pathlib.Path(os.environ["MW_OUT"])
repo = pathlib.Path(os.environ["MW_REPO"])
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
manifest = {
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "head_sha": sh(f'git -C "{repo}" rev-parse HEAD'),
    "head_subject": sh(f'git -C "{repo}" log --oneline -1'),
    "model": os.environ.get("MW_MODEL", ""),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"),
    "statuses": (pathlib.Path("/content/mw-status.env").read_text().strip().splitlines()
                 if pathlib.Path("/content/mw-status.env").exists() else []),
    "files": sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file()),
}
(out / "run-manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("manifest ok:", manifest["head_subject"])
PY

{
  echo "# mangaviewer colab run - $(date -u +%FT%TZ)"
  echo
  echo "## status"
  cat /content/mw-status.env 2>/dev/null || echo "(none)"
  echo
  echo "## analysis"
  for f in "$MW_OUT"/analysis/*/analysis.md; do
    [ -f "$f" ] || continue
    echo "### $(basename "$(dirname "$f")")"
    head -n 60 "$f"
    echo
  done
} > "$MW_OUT/SUMMARY.md"

tar -czf /content/results.tar.gz -C "$MW_OUT" --exclude=artifacts .
if [ -d "$MW_OUT/artifacts" ] && [ -n "$(ls -A "$MW_OUT/artifacts" 2>/dev/null)" ]; then
  tar -czf /content/artifacts.tar.gz -C "$MW_OUT" artifacts
fi
ls -la /content/results.tar.gz /content/artifacts.tar.gz 2>/dev/null

if [ -d /content/drive/MyDrive ]; then
  dest="/content/drive/MyDrive/mangaviewer-colab/$(date +%Y%m%d-%H%M%S)"
  mkdir -p "$dest"
  cp /content/results.tar.gz "$dest/"
  [ -f /content/artifacts.tar.gz ] && cp /content/artifacts.tar.gz "$dest/"
  echo "Drive copy: $dest"
else
  echo "Drive not mounted - browser download used"
fi

In [ ]:
import pathlib, shutil, subprocess, tempfile
from google.colab import files

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

pushed = False
if token:
    with tempfile.TemporaryDirectory() as td:
        bus = pathlib.Path(td) / "bus"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(bus)],
                       check=True, capture_output=True)

        def g(*args):
            return subprocess.run(["git", "-C", str(bus), *args],
                                  check=True, capture_output=True, text=True)

        have = subprocess.run(
            ["git", "-C", str(bus), "fetch", "--depth", "1", "origin", OUTBOX_BRANCH],
            capture_output=True, text=True).returncode == 0
        if have:
            g("checkout", "-q", "-B", OUTBOX_BRANCH, "FETCH_HEAD")
        else:
            g("checkout", "-q", "-B", OUTBOX_BRANCH)
        shutil.copy("/content/results.tar.gz", bus / "results.tar.gz")
        shutil.copy(f"{OUT_DIR}/SUMMARY.md", bus / "SUMMARY.md")
        try:
            # -f is required: the clone inherits the repo .gitignore (*.tar.gz, /*.md match these files)
            g("add", "-f", "results.tar.gz", "SUMMARY.md")
            g("-c", "user.email=colab-bot@users.noreply.github.com",
              "-c", "user.name=colab-bot", "commit", "-q", "-m", "colab results")
            push_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            r = subprocess.run(["git", "-C", str(bus), "push", "-q", push_url,
                                f"HEAD:{OUTBOX_BRANCH}"],
                               capture_output=True, text=True)
            pushed = r.returncode == 0
            print("outbox push:", "OK" if pushed else f"FAILED - {r.stderr[:300]}")
        except subprocess.CalledProcessError as exc:
            print("outbox push failed:", exc)
else:
    print("GITHUB_TOKEN 시크릿 없음 -> 아웃박스 푸시 생략 (Drive/다운로드 사용)")

if not CHAT_UI:
    subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server'", shell=True)
    print("vLLM 종료 (CHAT_UI=false)")

print("완료. 로컬 회수: colab/pull_results.ps1" + (" (아웃박스 푸시됨)" if pushed else ""))
files.download("/content/results.tar.gz")
if pathlib.Path("/content/artifacts.tar.gz").exists():
    files.download("/content/artifacts.tar.gz")

## 완료

- `results.tar.gz` — 분석/로그/매니페스트
- `artifacts.tar.gz` — APK (`adb install -r`로 에뮬에 설치)
- 로컬 회수: `colab/pull_results.ps1`
- 끝났으면 **런타임 -> 연결 해제** (A100은 연결돼 있는 동안 CCU가 계속 소모된다)